# Kentucky Lost-Wells Worker (KY)

This notebook processes exactly **701 latest-edition historical maps** for Kentucky.
It writes only well-symbol detections more than **100 m** from every loaded documented-well registry point. These are candidates, not confirmed undocumented wells.

Run only one worker per free Colab account. Every completed map is an atomic Drive checkpoint, so reopening and rerunning safely resumes.

## 1. Install the runtime

Choose **Runtime → Change runtime type → T4 GPU**, run this cell once, then choose **Runtime → Restart session**. A restart clears the old NumPy/Keras modules.

In [ ]:
import os, subprocess, sys, importlib.metadata as metadata
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["SM_FRAMEWORK"] = "tf.keras"

def run(command):
    print("RUN:", " ".join(command))
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT)
    print(result.stdout[-12000:])
    if result.returncode:
        raise RuntimeError(f"command failed with exit code {result.returncode}")

run(["apt-get", "-qq", "update"])
run(["apt-get", "-qq", "install", "-y", "gdal-bin", "libgdal-dev"])
tf_version = metadata.version("tensorflow")
print("Installed TensorFlow:", tf_version)
if not tf_version.startswith("2.20."):
    raise RuntimeError(f"Expected Colab TensorFlow 2.20.x, found {tf_version}; stop and report this")
run([sys.executable, "-m", "pip", "install", "--upgrade",
     "numpy==1.26.4", "opencv-python-headless==4.10.0.84"])
run([sys.executable, "-m", "pip", "install", "--upgrade", "tf_keras==2.20.1"])
run([sys.executable, "-m", "pip", "install", "--upgrade", "--no-deps",
     "keras_applications==1.0.8", "image-classifiers==1.0.0",
     "efficientnet==1.1.1", "segmentation-models==1.0.1"])
run([sys.executable, "-m", "pip", "install", "--upgrade",
     "simplekml==1.3.6", "geopandas", "requests"])
gdal_version = subprocess.check_output(["gdal-config", "--version"], text=True).strip()
run([sys.executable, "-m", "pip", "install", f"GDAL=={gdal_version}"])
print("Install complete. Use Runtime > Restart session before continuing.")

## 2. Mount Drive, configure paths, and update the code

Run this after the restart. Usually only `INPUT_ROOT` or `OUTPUT_ROOT` needs editing. Account 2 should make its shared input-folder shortcut appear as `MyDrive/lostwells_unet`.

In [ ]:
import os, subprocess, sys
from pathlib import Path
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["SM_FRAMEWORK"] = "tf.keras"

from google.colab import drive
drive.mount("/content/drive")

STATE = "KY"
INPUT_ROOT = Path("/content/drive/MyDrive/lostwells_unet")
OUTPUT_ROOT = Path("/content/drive/MyDrive/lostwells_unet_outputs")
REPO = Path("/content/lostwells")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/ShrishPremkrishna/lostwells.git", str(REPO)], check=True)

UNET = REPO / "UNET"
os.chdir(UNET)
print("state:", STATE)
print("inputs:", INPUT_ROOT)
print("outputs:", OUTPUT_ROOT)
print("code:", UNET)

## 3. Prerequisite and GPU gate

All four registries are required even for one state because quadrangles and the 100 m search can cross state borders.

In [ ]:
import json
import tensorflow as tf
import segmentation_models as sm
from osgeo import gdal

MODEL = INPUT_ROOT / "lbnl" / "unet_model.h5"
WELLS = [INPUT_ROOT / "wells" / f"{s}.geojson" for s in ("PA", "WV", "OH", "KY")]
MANIFEST = UNET / "manifests" / "core_appalachia_latest.csv"
STATE_OUT = OUTPUT_ROOT / STATE
STATE_OUT.mkdir(parents=True, exist_ok=True)

required = [MODEL, MANIFEST, *WELLS]
missing = [str(p) for p in required if not p.exists() or p.stat().st_size == 0]
if missing:
    raise FileNotFoundError("Missing required input(s):\n" + "\n".join(missing))
gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError("No GPU. Select a T4 runtime before inference.")
print("TensorFlow", tf.__version__, "GPU", gpus[0].name)
print("Input gate: PASS")

## 4. Progress before starting

An output counts as complete only when it is valid GeoJSON, including maps with zero candidates.

In [ ]:
import csv, json

with MANIFEST.open(newline="") as fh:
    state_rows = [r for r in csv.DictReader(fh) if r["primary_state"] == {
        "PA":"Pennsylvania", "WV":"West Virginia", "OH":"Ohio", "KY":"Kentucky"}[STATE]]

valid_outputs = []
candidate_count = 0
for path in STATE_OUT.glob("*_UOWs.geojson"):
    try:
        fc = json.loads(path.read_text())
        if fc.get("type") == "FeatureCollection":
            valid_outputs.append(path)
            candidate_count += len(fc.get("features", []))
    except Exception:
        pass

print(f"{STATE}: total maps={len(state_rows)}, completed={len(valid_outputs)}, "
      f"pending={len(state_rows)-len(valid_outputs)}, candidates so far={candidate_count}")

## 5. Run or resume full-state inference

Leave this cell running. The next map downloads in the background while the GPU processes the current map. Batch size starts at 32 and automatically halves after a GPU-memory error. If Colab disconnects, repeat sections 1–5; completed maps are skipped.

In [ ]:
command = [
    sys.executable, "scripts/run_batch.py",
    "--manifest", str(MANIFEST),
    "--model", str(MODEL),
    "--documented", *map(str, WELLS),
    "--out-dir", str(STATE_OUT),
    "--scratch", f"/content/lostwells_maps_{STATE}",
    "--states", STATE,
    "--batch-size", "32",
]
print("Starting/resuming", STATE)
subprocess.run(command, check=True)

## 6. State completion report and reversible 60 m deduplication

The original per-map files remain untouched. The merged file combines nearby repeated detections caused by overlapping map coverage.

In [ ]:
valid_outputs = []
candidate_count = 0
for path in STATE_OUT.glob("*_UOWs.geojson"):
    try:
        fc = json.loads(path.read_text())
        if fc.get("type") == "FeatureCollection":
            valid_outputs.append(path)
            candidate_count += len(fc.get("features", []))
    except Exception:
        pass

print(f"{STATE}: completed {len(valid_outputs)}/{len(state_rows)} maps; "
      f"raw candidates={candidate_count}")
failure_log = STATE_OUT / "failures.jsonl"
if failure_log.exists():
    failures = [line for line in failure_log.read_text().splitlines() if line.strip()]
    print(f"Logged failure attempts: {len(failures)} (reruns retry unfinished maps)")

merged = OUTPUT_ROOT / f"{STATE}_candidates_merged.geojson"
subprocess.run([sys.executable, "scripts/merge_detections.py",
                "--inputs", str(STATE_OUT), "--out", str(merged),
                "--radius-m", "60"], check=True)
print("Merged state output:", merged)